# Solara AI — Data Preprocessing Pipeline

**Goal:** Discover, remap, clean, and split solar panel fault datasets into 4 target classes.

**Target classes:** `dust`, `cracks`, `physical_damage`, `shading`  
**Optional reference:** `clean` (not a fault class, kept for reference)

**Output:** `ai-service/model-training/dataset/{train,val,test}/<class>/`

> **DISCLOSURE:** The Shading class contains synthetically generated images (simulated shadow overlays on clean panel photos), as no source dataset provided real shading examples. This should be disclosed in the project report and ideally supplemented with real images if available before final submission.

In [ ]:
import os
import sys
import shutil
import hashlib
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import imagehash
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 0

# Add parent so we can import config
sys.path.insert(0, str(Path.cwd().parent.parent))
from config import IMG_SIZE, BATCH_SIZE, FAULT_CLASSES, LABEL_MAP, RANDOM_SEED

RAW_DIR = Path.cwd().parent.parent / "dataset_raw"
OUT_DIR = Path.cwd().parent.parent / "dataset"
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.tif'}

print(f"RAW_DIR: {RAW_DIR}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"Target classes: {FAULT_CLASSES}")
print(f"Random seed: {RANDOM_SEED}")

## 1. Discovery Pass

Walk every folder inside `dataset_raw/` and print every leaf folder (containing images).  
For `source_1_afroz` and `source_3_pv_defect` which have train/val/test splits, we **flatten** them — treat all images in the same category as one pool.

In [ ]:
def discover_datasets(raw_dir):
    """Walk dataset_raw and find all leaf folders containing images."""
    discoveries = []
    
    for source_dir in sorted(raw_dir.iterdir()):
        if not source_dir.is_dir() or source_dir.name.startswith('.'):
            continue
        
        source_name = source_dir.name
        
        # Special handling: source_1 and source_3 have train/val/test nested
        # Flatten: walk recursively and find ALL leaf folders
        for root, dirs, files in os.walk(source_dir):
            root_path = Path(root)
            # Count image files in this exact folder
            img_files = [f for f in root_path.iterdir() 
                        if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
            
            if len(img_files) > 0:
                # This is a leaf folder with images
                rel_path = root_path.relative_to(raw_dir)
                category = root_path.name  # The innermost folder name
                
                # For source_1 and source_3, also note the split if present
                split = None
                parts = rel_path.parts
                if len(parts) > 2:  # e.g. source_1_afroz/train/Dusty
                    if parts[1] in ('train', 'val', 'test'):
                        split = parts[1]
                
                discoveries.append({
                    'source': source_name,
                    'rel_path': str(rel_path),
                    'folder_name': category,
                    'split': split,
                    'image_count': len(img_files),
                    'full_path': str(root_path)
                })
    
    return discoveries

discoveries = discover_datasets(RAW_DIR)

# Print discovery table
print("=" * 90)
print("DISCOVERY PASS — All leaf folders with images")
print("=" * 90)
print(f"{'Source':<25} {'Relative Path':<55} {'Images':>7}")
print("-" * 90)

total_images = 0
for d in sorted(discoveries, key=lambda x: (x['source'], x['rel_path'])):
    print(f"{d['source']:<25} {d['rel_path']:<55} {d['image_count']:>7}")
    total_images += d['image_count']

print("-" * 90)
print(f"TOTAL: {len(discoveries)} leaf folders, {total_images} images")
print("=" * 90)

### Flatten source_1_afroz and source_3_pv_defect

These sources have `train/val/test` splits from Kaggle. We **ignore** these splits — all images from the same category get pooled together. We'll do our own 70/15/15 split later.

In [ ]:
def flatten_source(discoveries, source_name):
    """For sources with train/val/test splits, pool images by category."""
    source_discoveries = [d for d in discoveries if d['source'] == source_name]
    
    # Group by folder_name (category)
    category_pool = defaultdict(list)
    for d in source_discoveries:
        category_pool[d['folder_name']].append(d)
    
    flattened = []
    for category, entries in sorted(category_pool.items()):
        total_imgs = sum(e['image_count'] for e in entries)
        paths = [e['full_path'] for e in entries]
        flattened.append({
            'source': source_name,
            'category': category,
            'total_images': total_imgs,
            'paths': paths,
            'note': f"Flattened from {len(entries)} splits" if len(entries) > 1 else "Direct"
        })
        print(f"  {source_name}/{category}: {total_imgs} images (from {len(entries)} location(s))")
    
    return flattened

print("Flattening source_1_afroz:")
source1_flat = flatten_source(discoveries, 'source_1_afroz')
print()
print("Flattening source_3_pv_defect:")
source3_flat = flatten_source(discoveries, 'source_3_pv_defect')
print()

# source_2 and source_4 are already flat (no train/val/test nesting)
print("source_2_dust and source_4_roboflow_crack: already flat, no flattening needed.")

## 2. Adaptive Class Remapping

Keyword-based mapping (case-insensitive) to our 4 target classes + clean + unmapped:

| Keywords | Target Class |
|---|---|
| dust, dusty, soil | `dust` |
| crack | `cracks` |
| damage (not electrical), physical, broken, delamination | `physical_damage` |
| shad, shadow, obstruct | `shading` |
| clean, normal, healthy, no fault | `clean` (reference, not a fault class) |
| Everything else | `UNMAPPED` |

In [ ]:
def classify_folder(folder_name):
    """Classify a folder name into target class using keyword matching."""
    name_lower = folder_name.lower().strip()
    
    # dust / dusty / soil
    if any(kw in name_lower for kw in ['dust', 'dusty', 'soil']):
        return 'dust', 'dust keyword'
    
    # crack
    if 'crack' in name_lower:
        return 'cracks', 'crack keyword'
    
    # physical damage (but not electrical)
    if any(kw in name_lower for kw in ['physical', 'broken', 'delamination']):
        return 'physical_damage', 'physical/broken/delamination keyword'
    if 'damage' in name_lower and 'electrical' not in name_lower:
        return 'physical_damage', 'damage keyword (non-electrical)'
    
    # shading
    if any(kw in name_lower for kw in ['shad', 'shadow', 'obstruct']):
        return 'shading', 'shading keyword'
    
    # clean / normal / healthy
    if any(kw in name_lower for kw in ['clean', 'normal', 'healthy', 'no fault']):
        return 'clean', 'clean/normal keyword'
    
    # UNMAPPED
    return 'UNMAPPED', 'no matching keyword'


# Build flat list of all category entries to classify
all_entries = []

# From flattened sources
for entry in source1_flat + source3_flat:
    all_entries.append(entry)

# From ALL flat sources (no train/val/test nesting)
# This catches source_2_dust, source_4_roboflow_crack, synthetic_shading, etc.
flat_sources_seen = set()
for d in discoveries:
    if d['split'] is None and d['source'] not in flat_sources_seen:
        flat_sources_seen.add(d['source'])
        # Collect all leaf folders for this source
        source_entries = [x for x in discoveries if x['source'] == d['source'] and x['split'] is None]
        for entry in source_entries:
            all_entries.append({
                'source': entry['source'],
                'category': entry['folder_name'],
                'total_images': entry['image_count'],
                'paths': [entry['full_path']],
                'note': 'Direct'
            })

# Now classify each entry
remap_results = []
for entry in all_entries:
    target_class, keyword = classify_folder(entry['category'])
    remap_results.append({
        'source': entry['source'],
        'original_category': entry['category'],
        'total_images': entry['total_images'],
        'target_class': target_class,
        'matched_keyword': keyword,
        'paths': entry['paths'],
        'note': entry.get('note', '')
    })

# Print audit table
print("=" * 100)
print("REMAPPING AUDIT TABLE")
print("=" * 100)
print(f"{'Source':<25} {'Original Category':<25} {'Keyword Match':<30} {'Target Class':<20} {'Images':>7}")
print("-" * 100)

for r in sorted(remap_results, key=lambda x: (x['target_class'], x['source'])):
    print(f"{r['source']:<25} {r['original_category']:<25} {r['matched_keyword']:<30} {r['target_class']:<20} {r['total_images']:>7}")

print("-" * 100)
print(f"TOTAL entries: {len(remap_results)}")
print("=" * 100)

In [ ]:
# Show unmapped entries clearly
unmapped = [r for r in remap_results if r['target_class'] == 'UNMAPPED']
clean_ref = [r for r in remap_results if r['target_class'] == 'clean']

print("=" * 70)
print("UNMAPPED FOLDERS (not mapped to any of the 4 fault classes)")
print("=" * 70)
if unmapped:
    for u in unmapped:
        print(f"  {u['source']}/{u['original_category']}: {u['total_images']} images")
        print(f"    Note: {u['note']}")
else:
    print("  None — all folders mapped successfully.")
print()

print("=" * 70)
print("CLEAN REFERENCE (not a fault class, kept for reference)")
print("=" * 70)
if clean_ref:
    for c in clean_ref:
        print(f"  {c['source']}/{c['original_category']}: {c['total_images']} images")
else:
    print("  None found.")
print()

# Count per target class
class_counts = Counter()
for r in remap_results:
    if r['target_class'] != 'UNMAPPED':
        class_counts[r['target_class']] += r['total_images']

print("=" * 70)
print("IMAGE COUNT PER TARGET CLASS (before cleaning)")
print("=" * 70)
for cls in FAULT_CLASSES + ['clean']:
    count = class_counts.get(cls, 0)
    label = LABEL_MAP.get(cls, cls)
    print(f"  {label:<20} {count:>7} images")
print(f"  {'UNMAPPED':<20} {sum(u['total_images'] for u in unmapped):>7} images")

## 3. Shading Class Gap Check

In [ ]:
shading_count = class_counts.get('shading', 0)

print("=" * 70)
print(f"SHADING CLASS: {shading_count} images found after remapping")
print("=" * 70)

if shading_count < 100:
    print(f"\n⚠️  WARNING: Shading class has only {shading_count} images (under 100).")
    print("Recommendations:")
    print("  (a) Manually collect additional shading/obstruction images")
    print("  (b) Apply heavier augmentation specifically for this class during training")
    print("  (c) Check the UNMAPPED bucket for anything that could be relabeled as shading")
    print("  (d) Consider reducing to 3 classes if shading data is insufficient")
elif shading_count < 200:
    print(f"\n⚠️  CAUTION: Shading class has {shading_count} images (under 200).")
    print("Consider collecting more images or applying heavier augmentation.")
else:
    print(f"\n✅ Shading class has sufficient images ({shading_count}).")

## 4. Data Cleaning

- Remove corrupted/unreadable images
- Remove duplicates via perceptual hashing (imagehash)
- Remove images below 100x100px
- Collect all valid file paths per target class

In [ ]:
def collect_image_paths(remap_results, target_class):
    """Collect all image file paths for a target class from all sources."""
    paths = []
    for r in remap_results:
        if r['target_class'] != target_class:
            continue
        for p in r['paths']:
            p = Path(p)
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                paths.append(p)
            elif p.is_dir():
                # It's a directory — walk it
                for img in p.rglob('*'):
                    if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
                        paths.append(img)
    return paths

# Collect paths for each target class
class_paths = {}
for cls in FAULT_CLASSES:
    class_paths[cls] = collect_image_paths(remap_results, cls)
    print(f"{LABEL_MAP[cls]:<20}: {len(class_paths[cls]):>5} images collected")

total_before = sum(len(v) for v in class_paths.values())
print(f"\nTotal before cleaning: {total_before} images")

In [ ]:
MIN_SIZE = 100  # Minimum dimension in pixels

def is_valid_image(path):
    """Check if image can be opened and meets minimum size."""
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            w, h = img.size
            if w < MIN_SIZE or h < MIN_SIZE:
                return False, f"Too small ({w}x{h})"
        return True, "OK"
    except Exception as e:
        return False, f"Corrupt: {e}"

def compute_phash(path):
    """Compute perceptual hash for duplicate detection."""
    try:
        with Image.open(path) as img:
            return imagehash.phash(img)
    except:
        return None

def deduplicate_by_phash(paths):
    """Remove duplicate images using perceptual hashing."""
    seen_hashes = {}
    unique_paths = []
    duplicates = 0
    
    for p in paths:
        h = compute_phash(p)
        if h is None:
            unique_paths.append(p)
            continue
        
        # Check if we've seen a similar hash (hamming distance <= 5)
        is_dup = False
        for seen_h in seen_hashes.values():
            if h - seen_h <= 5:
                is_dup = True
                break
        
        if not is_dup:
            seen_hashes[str(p)] = h
            unique_paths.append(p)
        else:
            duplicates += 1
    
    return unique_paths, duplicates

print("Cleaning functions defined.")

In [ ]:
cleaned_paths = {}
stats = {}

for cls in FAULT_CLASSES:
    paths = class_paths[cls]
    before_count = len(paths)
    removed_corrupt = 0
    removed_small = 0
    
    # Step 1: Remove corrupt/too-small images
    valid_paths = []
    for p in paths:
        valid, reason = is_valid_image(p)
        if valid:
            valid_paths.append(p)
        else:
            removed_corrupt += 1
    
    # Step 2: Deduplicate
    deduped_paths, removed_dupes = deduplicate_by_phash(valid_paths)
    
    cleaned_paths[cls] = deduped_paths
    after_count = len(deduped_paths)
    
    stats[cls] = {
        'before': before_count,
        'corrupt': removed_corrupt,
        'duplicates': removed_dupes,
        'after': after_count
    }
    
    print(f"{LABEL_MAP[cls]:<20}: {before_count:>5} → {after_count:>5} "
          f"(removed: {removed_corrupt} corrupt, {removed_dupes} duplicates)")

total_after = sum(len(v) for v in cleaned_paths.values())
print(f"\nTotal: {total_before} → {total_after} images after cleaning")

## 5. Class Balance Report

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

classes = []
counts = []
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

for cls in FAULT_CLASSES:
    classes.append(LABEL_MAP[cls])
    counts.append(len(cleaned_paths[cls]))

bars = ax.bar(classes, counts, color=colors, edgecolor='black', linewidth=0.5)

# Add count labels on bars
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)

# Flag underrepresented classes
for i, (cls, count) in enumerate(zip(FAULT_CLASSES, [len(cleaned_paths[c]) for c in FAULT_CLASSES])):
    if count < 200:
        ax.text(i, count/2, '⚠️ UNDER', ha='center', va='center', 
                fontsize=10, color='white', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.7))

ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Class Balance After Cleaning', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.savefig('class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClass balance report:")
for cls, count in zip(FAULT_CLASSES, counts):
    status = "⚠️  UNDERREPRESENTED (<200)" if count < 200 else "✅ OK"
    print(f"  {LABEL_MAP[cls]:<20} {count:>5} images  {status}")

## 6. Train/Val/Test Split

Stratified 70/15/15 split per target class. Uses `RANDOM_SEED=42` from config.

In [ ]:
from sklearn.model_selection import train_test_split

def stratified_split(paths, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=RANDOM_SEED):
    """Stratified split (for single class, just random split)."""
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    
    # First split: train vs (val + test)
    train_paths, temp_paths = train_test_split(paths, test_size=(val_ratio + test_ratio), 
                                                random_state=seed)
    
    # Second split: val vs test
    relative_test = test_ratio / (val_ratio + test_ratio)
    val_paths, test_paths = train_test_split(temp_paths, test_size=relative_test, 
                                              random_state=seed)
    
    return train_paths, val_paths, test_paths

# Create output directories and copy files
split_counts = {}

for cls in FAULT_CLASSES:
    paths = cleaned_paths[cls]
    if len(paths) == 0:
        print(f"  {LABEL_MAP[cls]}: SKIPPED (no images)")
        continue
    
    train_paths, val_paths, test_paths = stratified_split(paths)
    
    split_counts[cls] = {
        'train': len(train_paths),
        'val': len(val_paths),
        'test': len(test_paths)
    }
    
    # Create directories and copy
    for split_name, split_paths in [('train', train_paths), ('val', val_paths), ('test', test_paths)]:
        split_dir = OUT_DIR / split_name / cls
        split_dir.mkdir(parents=True, exist_ok=True)
        
        for src_path in split_paths:
            dst_path = split_dir / src_path.name
            # Handle duplicate filenames
            if dst_path.exists():
                stem = dst_path.stem
                suffix = dst_path.suffix
                counter = 1
                while dst_path.exists():
                    dst_path = split_dir / f"{stem}_{counter}{suffix}"
                    counter += 1
            shutil.copy2(src_path, dst_path)
    
    print(f"  {LABEL_MAP[cls]:<20}: {len(train_paths):>4} train | {len(val_paths):>4} val | {len(test_paths):>4} test")

print("\nSplit complete. Output directories:")
for split in ['train', 'val', 'test']:
    total = sum(split_counts.get(cls, {}).get(split, 0) for cls in FAULT_CLASSES)
    print(f"  {OUT_DIR / split}: {total} images")

In [ ]:
# Final counts table
print("=" * 75)
print("FINAL DATASET COUNTS (after split)")
print("=" * 75)
print(f"{'Class':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 75)

grand_totals = {'train': 0, 'val': 0, 'test': 0}
for cls in FAULT_CLASSES:
    if cls in split_counts:
        t = split_counts[cls]['train']
        v = split_counts[cls]['val']
        te = split_counts[cls]['test']
        total = t + v + te
        print(f"{LABEL_MAP[cls]:<20} {t:>8} {v:>8} {te:>8} {total:>8}")
        grand_totals['train'] += t
        grand_totals['val'] += v
        grand_totals['test'] += te

print("-" * 75)
grand_total = sum(grand_totals.values())
print(f"{'TOTAL':<20} {grand_totals['train']:>8} {grand_totals['val']:>8} {grand_totals['test']:>8} {grand_total:>8}")
print("=" * 75)

## 7. Preprocessing Functions

These functions will be reused in `ai-service/app/preprocessing.py` for the live API:
- **Letterbox resize** to 224x224 (preserve aspect ratio, pad with black)
- **CLAHE contrast enhancement** on luminance channel
- No augmentation here — augmentation happens live during training

In [ ]:
import cv2

def letterbox_resize(img_array, target_size=(224, 224)):
    """Resize image preserving aspect ratio, pad with black."""
    h, w = img_array.shape[:2]
    target_w, target_h = target_size
    
    # Calculate scale to fit within target
    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    
    # Resize
    resized = cv2.resize(img_array, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # Create black canvas and center
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
    
    return canvas

def apply_clahe(img_array):
    """Apply CLAHE contrast enhancement on luminance channel."""
    # Convert to LAB color space
    lab = cv2.cvtColor(img_array, cv2.COLOR_RGB2LAB)
    l_channel = lab[:, :, 0]
    
    # Apply CLAHE to L channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(l_channel)
    
    # Convert back to RGB
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return enhanced

def preprocess_image(img_path, target_size=(224, 224)):
    """Full preprocessing pipeline for a single image."""
    # Read image
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Letterbox resize
    img = letterbox_resize(img, target_size)
    
    # CLAHE enhancement
    img = apply_clahe(img)
    
    return img

print("Preprocessing functions defined.")
print("  - letterbox_resize: preserve aspect ratio, pad to 224x224")
print("  - apply_clahe: CLAHE contrast enhancement on luminance")
print("  - preprocess_image: full pipeline (read → resize → CLAHE)")

## 8. Sanity Check Visualization

4x4 grid of random preprocessed samples per target class.

In [ ]:
np.random.seed(RANDOM_SEED)

for cls in FAULT_CLASSES:
    paths = cleaned_paths[cls]
    if len(paths) == 0:
        print(f"{LABEL_MAP[cls]}: SKIPPED (no images)")
        continue
    
    # Sample up to 16 random images
    sample_paths = np.random.choice(paths, size=min(16, len(paths)), replace=False)
    
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    fig.suptitle(f'{LABEL_MAP[cls]} — Preprocessed Samples', fontsize=16, fontweight='bold')
    
    for i, ax in enumerate(axes.flat):
        if i < len(sample_paths):
            img = preprocess_image(sample_paths[i])
            if img is not None:
                ax.imshow(img)
                ax.set_title(Path(sample_paths[i]).name[:20], fontsize=8)
            else:
                ax.text(0.5, 0.5, 'FAILED', ha='center', va='center', color='red')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(f'sanity_check_{cls}.png', dpi=150, bbox_inches='tight')
    plt.show()

print("Sanity check complete.")

## Summary

### What was done:
1. **Discovery** — walked all source folders, printed every leaf folder with image counts
2. **Flattening** — pooled source_1 and source_3's train/val/test splits into single category pools
3. **Adaptive remapping** — keyword-based classification into 4 target classes + clean + unmapped
4. **Shading gap check** — flagged if shading class is underrepresented
5. **Cleaning** — removed corrupt images, duplicates (perceptual hash), and too-small images
6. **Class balance** — bar chart with underrepresentation flags
7. **Stratified split** — 70/15/15 train/val/test per class
8. **Preprocessing functions** — letterbox resize + CLAHE (reused in live API)
9. **Sanity check** — visual confirmation of preprocessed samples

### Output directories:
```
ai-service/model-training/dataset/
├── train/
│   ├── dust/
│   ├── cracks/
│   ├── physical_damage/
│   └── shading/
├── val/
│   ├── dust/
│   ├── cracks/
│   ├── physical_damage/
│   └── shading/
└── test/
    ├── dust/
    ├── cracks/
    ├── physical_damage/
    └── shading/
```